# Eigenvalues and eigenvectors for full weight matrix

Load the four group-to-group weight matrices (E→E, E→I, I→E, I→I), build the full block weight matrix, and compute its eigenvalues and eigenvectors.

In [152]:
from pathlib import Path

import numpy as np

from buryn_utils import weight_utils

In [153]:
BURYNOUTPUTS_DIR = '../data/buryn_data/buryn_outputs_0'
WEIGHTS_E_TO_E_FILEPATH = Path(BURYNOUTPUTS_DIR+'/weights/weights_step0.txt')
WEIGHTS_E_TO_I_FILEPATH = Path(BURYNOUTPUTS_DIR+'/weights/weights_e_to_i_step0.txt')
WEIGHTS_I_TO_E_FILEPATH = Path(BURYNOUTPUTS_DIR+'/weights/weights_i_to_e_step0.txt')
WEIGHTS_I_TO_I_FILEPATH = Path(BURYNOUTPUTS_DIR+'/weights/weights_i_to_i_step0.txt')
g_total = 51*10**(-9) # Siemens

weights_filepaths = [
    WEIGHTS_E_TO_E_FILEPATH,
    WEIGHTS_E_TO_I_FILEPATH,
    WEIGHTS_I_TO_E_FILEPATH,
    WEIGHTS_I_TO_I_FILEPATH,
]
for weights_filepath in weights_filepaths:
    if not weights_filepath.exists():
        raise FileNotFoundError(f'Weights file not found: {weights_filepath}')

In [154]:
def infer_shape(weights):
    pre_ids = [entry[0] for entry in weights]
    post_ids = [entry[1] for entry in weights]
    return max(pre_ids) + 1, max(post_ids) + 1

def build_weight_matrix(weights, shape):
    matrix = np.zeros(shape, dtype=float)
    for pre_id, post_id, weight in weights:
        matrix[pre_id, post_id] = weight
    return matrix

weights_e_to_e = weight_utils.load_weights(WEIGHTS_E_TO_E_FILEPATH)
weights_e_to_i = weight_utils.load_weights(WEIGHTS_E_TO_I_FILEPATH)
weights_i_to_e = weight_utils.load_weights(WEIGHTS_I_TO_E_FILEPATH)
weights_i_to_i = weight_utils.load_weights(WEIGHTS_I_TO_I_FILEPATH)

shape_e_to_e = infer_shape(weights_e_to_e)
shape_e_to_i = infer_shape(weights_e_to_i)
shape_i_to_e = infer_shape(weights_i_to_e)
shape_i_to_i = infer_shape(weights_i_to_i)

num_e_pre, num_e_post = shape_e_to_e
num_e_pre_alt, num_i_post = shape_e_to_i
num_i_pre, num_e_post_alt = shape_i_to_e
num_i_pre_alt, num_i_post_alt = shape_i_to_i

if num_e_pre != num_e_pre_alt:
    raise ValueError('E pre-synaptic counts mismatch between E→E and E→I weights.')
if num_i_pre != num_i_pre_alt:
    raise ValueError('I pre-synaptic counts mismatch between I→E and I→I weights.')
if num_e_post != num_e_post_alt:
    raise ValueError('E post-synaptic counts mismatch between E→E and I→E weights.')
if num_i_post != num_i_post_alt:
    raise ValueError('I post-synaptic counts mismatch between E→I and I→I weights.')

weights_e_to_e_matrix = build_weight_matrix(weights_e_to_e, shape_e_to_e)
weights_e_to_i_matrix = build_weight_matrix(weights_e_to_i, shape_e_to_i)
weights_i_to_e_matrix = build_weight_matrix(weights_i_to_e, shape_i_to_e)
weights_i_to_i_matrix = build_weight_matrix(weights_i_to_i, shape_i_to_i)

full_weight_matrix = np.block(
    [
        [weights_e_to_e_matrix, weights_e_to_i_matrix],
        [weights_i_to_e_matrix*-1.0, weights_i_to_i_matrix*-1.0],
    ]
)

full_weight_matrix.shape

(2000, 2000)

In [155]:
eigenvalues, eigenvectors = np.linalg.eig(full_weight_matrix)

sorted_indices = np.argsort(np.abs(eigenvalues))[::-1]
top_k = 10

print(f'Computed {len(eigenvalues)} eigenvalues for the full matrix.')
print(f'Top {top_k} eigenvalues by magnitude:')
for idx in sorted_indices[:top_k]:
    eigenvalue = eigenvalues[idx]
    print(f'  {idx}: {eigenvalue}, {eigenvalue/g_total}')

Computed 2000 eigenvalues for the full matrix.
Top 10 eigenvalues by magnitude:
  0: (2.7941574256253558e-09+9.542123281409765e-08j), (0.05478740050245796+1.8710045649823066j)
  1: (2.7941574256253558e-09-9.542123281409765e-08j), (0.05478740050245796-1.8710045649823066j)
  2: (1.5868145626880002e-09+8.51021896721624e-09j), (0.031114011033098042+0.16686703857286744j)
  3: (1.5868145626880002e-09-8.51021896721624e-09j), (0.031114011033098042-0.16686703857286744j)
  6: (1.608076842956261e-09+8.173533866829013e-09j), (0.03153091848933845+0.16026536993782378j)
  7: (1.608076842956261e-09-8.173533866829013e-09j), (0.03153091848933845-0.16026536993782378j)
  4: (1.7302984661699436e-09+8.116891020832189e-09j), (0.03392742090529301+0.15915472589867036j)
  5: (1.7302984661699436e-09-8.116891020832189e-09j), (0.03392742090529301-0.15915472589867036j)
  8: (1.5700796578138164e-09+8.064307318889669e-09j), (0.030785875643408164+0.15812367291940527j)
  9: (1.5700796578138164e-09-8.064307318889669e-09